# EXP-015: PLR-ResMLP only (RealMLP의 PLR + ResMLP의 Residual)

**근거:** RealMLP는 **입력 표현 (PLR)**, ResMLP는 **본체 (Residual)** 강화 — 직교 기법. 합치면 더 큰 표현력 가능.

**변경점 vs EXP-009 PLR-MLP:**
- 본체 `[256, 128, 64]` 점진 축소 → **`input → 3 × ResidualBlock(256) → 1`** (일정 width + skip connection)
- 그 외 동일 (PLR encoder, embedding, 학습 설정)

**ResidualBlock:**
```
x → Linear(256, 256) → ReLU → Dropout → Linear(256, 256) → +x → BatchNorm → ReLU
```

**비교 지표**: MLP solo OOF — EXP-009 PLR-MLP 0.94486과 직접 비교.

In [1]:
import pandas as pd
import numpy as np
import time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
NUM_COLS = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]

N_FOLDS, SEED = 5, 42
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}')

Train: (439140, 16), Test: (188165, 15)


## 1. PLR encoder + ResidualBlock + PLR-ResMLP

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

class PLREncoder(nn.Module):
    """Periodic Linear Representation for numeric features (RealMLP-style)."""
    def __init__(self, n_numeric, k=16, hidden=8, sigma=2.33):
        super().__init__()
        self.freqs = nn.Parameter(torch.randn(n_numeric, k) * sigma)
        self.linear = nn.Linear(2 * k, hidden)
        self.act = nn.GELU()
    def forward(self, x_num):
        z = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs.unsqueeze(0)
        p = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)
        out = self.act(self.linear(p))
        return out.flatten(start_dim=1)


class ResidualBlock(nn.Module):
    """ResNet-style residual block for tabular MLP (skip connection).

    Pattern (from external nn-residual-network notebook):
        x → Linear → ReLU → Dropout → Linear → +x → BatchNorm → ReLU
    """
    def __init__(self, hidden_dim, dropout=0.2):
        super().__init__()
        self.linear1 = nn.Linear(hidden_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.bn = nn.BatchNorm1d(hidden_dim)
    def forward(self, x):
        residual = x
        out = self.linear1(x)
        out = self.act(out)
        out = self.dropout(out)
        out = self.linear2(out)
        out = out + residual          # skip connection
        out = self.bn(out)
        out = self.act(out)
        return out


class PLRResMLP(nn.Module):
    def __init__(self, cat_cards, emb_dims, n_numeric, plr_k=16, plr_hidden=8,
                 hidden_dim=256, n_blocks=3, dropout=0.2):
        super().__init__()
        # Cat embeddings (EXP-009과 동일)
        self.embs = nn.ModuleList([nn.Embedding(card, dim) for card, dim in zip(cat_cards, emb_dims)])
        emb_total = sum(emb_dims)
        # Raw numeric BN
        self.bn_num = nn.BatchNorm1d(n_numeric)
        # PLR encoder
        self.plr = PLREncoder(n_numeric, k=plr_k, hidden=plr_hidden, sigma=2.33)
        plr_total = n_numeric * plr_hidden
        in_dim = emb_total + n_numeric + plr_total

        # Input layer: projection to hidden_dim
        self.input_layer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # Stack of residual blocks
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)
        ])
        # Output layer
        self.output_layer = nn.Linear(hidden_dim, 1)

        print(f'PLRResMLP: in_dim={in_dim} (emb {emb_total} + raw_num {n_numeric} + plr {plr_total}), '
              f'hidden_dim={hidden_dim}, n_blocks={n_blocks}')

    def forward(self, x_cat, x_num):
        emb = torch.cat([emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embs)], dim=1)
        raw_num = self.bn_num(x_num)
        plr_feat = self.plr(x_num)
        x = torch.cat([emb, raw_num, plr_feat], dim=1)
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x).squeeze(1)

device: cuda


## 2. Cat / Num indexing

In [3]:
cat_cards = []
X_cat_tr = np.zeros((len(X_le), len(CAT_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(X_test_le), len(CAT_COLS)), dtype=np.int64)
for i, c in enumerate(CAT_COLS):
    n_unique = X_le[c].max() + 1
    cat_cards.append(n_unique + 1)
    X_cat_tr[:, i] = X_le[c].values + 1
    te_vals = X_test_le[c].values + 1
    te_vals[te_vals < 0] = 0
    X_cat_te[:, i] = te_vals

X_num_tr_raw = X_le[NUM_COLS].values.astype(np.float32)
X_num_te_raw = X_test_le[NUM_COLS].values.astype(np.float32)

## 3. 5-fold 학습 (EXP-009 동일 설정)

In [4]:
EMB_DIMS = [16, 4, 8]
BATCH_SIZE = 4096
MAX_EPOCHS = 80
PATIENCE = 20

oof_mlp = np.zeros(len(X_le))
test_mlp = np.zeros(len(X_test_le))
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_num_tr_raw[tr_idx])
    X_num_va = scaler.transform(X_num_tr_raw[va_idx])
    X_num_te = scaler.transform(X_num_te_raw)

    Xc_tr = torch.tensor(X_cat_tr[tr_idx], dtype=torch.long)
    Xn_tr = torch.tensor(X_num_tr, dtype=torch.float32)
    yt_tr = torch.tensor(y[tr_idx], dtype=torch.float32)
    Xc_va = torch.tensor(X_cat_tr[va_idx], dtype=torch.long, device=device)
    Xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
    yt_va = y[va_idx]
    Xc_te = torch.tensor(X_cat_te, dtype=torch.long, device=device)
    Xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)

    train_loader = DataLoader(
        TensorDataset(Xc_tr, Xn_tr, yt_tr),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        num_workers=0, pin_memory=True,
    )

    torch.manual_seed(SEED)
    model = PLRResMLP(cat_cards, EMB_DIMS, len(NUM_COLS),
                     hidden_dim=256, n_blocks=3, dropout=0.2).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=MAX_EPOCHS)
    bce = nn.BCEWithLogitsLoss()

    best_auc = -1
    best_state = None
    wait = 0
    epochs_used = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for xc, xn, yb in train_loader:
            xc = xc.to(device, non_blocking=True)
            xn = xn.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            logit = model(xc, xn)
            loss = bce(logit, yb)
            loss.backward()
            optim.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            va_logit = model(Xc_va, Xn_va).cpu().numpy()
        va_prob = 1 / (1 + np.exp(-va_logit))
        auc = roc_auc_score(yt_va, va_prob)
        epochs_used = epoch + 1

        if auc > best_auc + 1e-6:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        va_logit = model(Xc_va, Xn_va).cpu().numpy()
        te_logit = model(Xc_te, Xn_te).cpu().numpy()
    oof_mlp[va_idx] = 1 / (1 + np.exp(-va_logit))
    test_mlp += (1 / (1 + np.exp(-te_logit))) / N_FOLDS
    early_stopped = epochs_used < MAX_EPOCHS
    print(f'PLR-ResMLP fold {fold}: best AUC={best_auc:.5f}, epochs_used={epochs_used} {"(ES)" if early_stopped else "(max)"}')

    del model, optim, sched, Xc_va, Xn_va, Xc_te, Xn_te
    torch.cuda.empty_cache()

mlp_oof_auc = roc_auc_score(y, oof_mlp)
print(f'\nPLR-ResMLP OOF AUC: {mlp_oof_auc:.5f}')
print(f'  vs EXP-009 PLR-MLP    0.94486: {mlp_oof_auc-0.94486:+.5f}')
print(f'  vs EXP-012 PLR+inner  0.94763: {mlp_oof_auc-0.94763:+.5f}')
print(f'  elapsed: {time.time()-t0:.1f}s')

PLRResMLP: in_dim=127 (emb 28 + raw_num 11 + plr 88), hidden_dim=256, n_blocks=3
PLR-ResMLP fold 0: best AUC=0.94463, epochs_used=31 (ES)
PLRResMLP: in_dim=127 (emb 28 + raw_num 11 + plr 88), hidden_dim=256, n_blocks=3
PLR-ResMLP fold 1: best AUC=0.94238, epochs_used=27 (ES)
PLRResMLP: in_dim=127 (emb 28 + raw_num 11 + plr 88), hidden_dim=256, n_blocks=3
PLR-ResMLP fold 2: best AUC=0.94382, epochs_used=29 (ES)
PLRResMLP: in_dim=127 (emb 28 + raw_num 11 + plr 88), hidden_dim=256, n_blocks=3
PLR-ResMLP fold 3: best AUC=0.94261, epochs_used=29 (ES)
PLRResMLP: in_dim=127 (emb 28 + raw_num 11 + plr 88), hidden_dim=256, n_blocks=3
PLR-ResMLP fold 4: best AUC=0.94381, epochs_used=30 (ES)

PLR-ResMLP OOF AUC: 0.94341
  vs EXP-009 PLR-MLP    0.94486: -0.00145
  vs EXP-012 PLR+inner  0.94763: -0.00422
  elapsed: 470.4s


## 4. Save predictions + 결론

In [5]:
np.save('../submissions/exp015_plr_resmlp_oof.npy', oof_mlp)
np.save('../submissions/exp015_plr_resmlp_test.npy', test_mlp)
sub = pd.DataFrame({'id': test['id'], 'PitNextLap': test_mlp})
sub.to_csv('../submissions/submission_exp015_plr_resmlp.csv', index=False)
print(f'saved exp015 OOF/test arrays + submission CSV')
print(f'test pred mean: {test_mlp.mean():.4f}')
print()
print('=== EXP-015 결론 ===')
print(f'  EXP-009 PLR-MLP solo (no residual) : 0.94486')
print(f'  EXP-012 PLR-MLP + inner ens=3      : 0.94763')
print(f'  EXP-015 PLR-ResMLP solo            : {mlp_oof_auc:.5f}')
print()
delta_vs_009 = mlp_oof_auc - 0.94486
if delta_vs_009 > 0.003:
    print(f'  → Residual 효과 큼 (+{delta_vs_009:.5f} vs PLR-MLP). EXP-016 4-way 풀 학습 + ResMLP 추가 가치 큼')
elif delta_vs_009 > 0.0005:
    print(f'  → Residual 효과 중간 (+{delta_vs_009:.5f}). EXP-016 4-way 학습 가치')
elif delta_vs_009 > -0.0005:
    print(f'  → Residual 효과 미미. PLR-MLP과 거의 동일')
else:
    print(f'  → Residual 부정. PLR-MLP가 더 나음')

saved exp015 OOF/test arrays + submission CSV
test pred mean: 0.2086

=== EXP-015 결론 ===
  EXP-009 PLR-MLP solo (no residual) : 0.94486
  EXP-012 PLR-MLP + inner ens=3      : 0.94763
  EXP-015 PLR-ResMLP solo            : 0.94341

  → Residual 부정. PLR-MLP가 더 나음
